In [ ]:
import getpass
import psycopg

# Obtain a database connection
db_host = 'localhost'
db_user = 'postgres'
db_name = 'securities_master'
db_pass = getpass.getpass('db password')



In [ ]:
import pandas as pd
import numpy as np
import mplfinance as mpl 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
#con = psycopg.connect(f"host={db_host} user={db_user} password={db_pass} dbname={db_name}") 
with con:
    pass


In [ ]:
train = GOOGLE.loc['2023-01-30':'2023-02-10']
test =  GOOGLE.loc['2023-02-11':'2023-02-20']
         
mpl.plot(train, type='line')
mpl.plot(test, type='line')

In [ ]:
def create_dataset(dataset, lookback):
    """Transform a time series into a prediction dataset
    
    Args:
        dataset: A numpy array of time series, first dimension is the time steps
        lookback: Size of window for prediction
    """
    X, y = [], []
    for i in range(len(dataset)-lookback):
        feature = dataset.iloc[i:i+lookback]
        target = dataset.iloc[i+1:i+lookback+1]
        X.append(feature)
        y.append(target)
    return torch.tensor(X).type(torch.float32), torch.tensor(y).type(torch.float32)

In [ ]:
class basic_model(nn.Module):
    def __init__(self,lookback,hidden_size=50,num_layers=1,dropout=0):
        super().__init__()
        self.lstm = nn.LSTM(input_size=lookback, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.linear = nn.Linear(hidden_size, 1)
    def forward(self, x):
        x,_ = self.lstm(x)
        x = self.linear(x)
        return x

In [ ]:
lookback = 4
train_x, train_y = create_dataset(train.open,lookback)
test_x, test_y = create_dataset(test.open,lookback)

print(train_x.shape,train_y.shape)
print(test_x.shape,test_y.shape)
train_x[0]

In [ ]:
n_epochs = 500
batch_size = 32

loader = data.DataLoader(data.TensorDataset(train_x.to(device), train_y.to(device)), shuffle=True, batch_size=batch_size)
model = basic_model(lookback,20,1).to(device)
optimiser = optim.Adam(model.parameters())
loss_fn = nn.MSELoss()


for epoch in range(n_epochs):
    
    train_err = 0
    model.train()
    
    for x_batch, y_batch in loader:
        y_pred = model(x_batch)
        loss = loss_fn(y_pred, y_batch)
                
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        
    #Validation
    if epoch % 50 != 0:
        continue
    
    model.eval()
    with torch.no_grad():
        y_pred = model(train_x.to(device))
        train_rmse = np.sqrt(loss_fn(y_pred.to('cpu'), train_y))
                             
        y_pred = model(test_x.to(device))
        test_rmse = np.sqrt(loss_fn(y_pred.to('cpu'), test_y))
        
    print("Epoch %d: train RMSE %.4f, test RMSE %.4f" % (epoch, train_rmse, test_rmse))

In [ ]:
predictions = model(test_x.to(device)).to('cpu').detach().numpy()

In [ ]:
x,y =  test.index, test.open.values

plt.plot(x,y, color='b')
plt.plot(x[lookback:],predictions,color='c')

In [ ]:
test2 =  GOOGLE.loc['2024-02-11':'2024-05-20']
test2_x, test_y = create_dataset(test2.open,lookback)

In [ ]:
plt.plot(test2.index, test2.open.values, color='b')
plt.plot(test2.index[lookback:],model(test2_x.to(device)).to('cpu').detach().numpy(),color='c')